In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/kv-cache")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# KV Cache from Scratch — Practice

**Tier: T0.** numpy and matplotlib only, on a laptop or a Colab CPU runtime; no GPU and no PyTorch. The random
weights and the reference formula come from [`kernel-core`](../kernel-core/README.md) (`kerncore.kv`). Sizes are
printed in binary units (KiB, GiB) with decimal GB in brackets, as in the [primer](kv-cache-primer.md).

Fill in the four blanks (**A–D**), each marked `# YOUR CODE HERE`, then run the test cells below each section.
Each blank raises `NotImplementedError` until you replace it, so a test that reaches an unwritten
blank stops and names it; a wrong answer stops with an `AssertionError`; a right one
prints `PASSED`.

**The four blanks:**
- **A** — grow the cache (concatenate new K, V onto the past)
- **B** — the causal mask
- **C** — the single decode step (feed one token + the cache)
- **D** — the cache-size formula

Everything else (the model, naive generation, the tests, plotting) is provided so you can
stay focused on the caching logic. Stuck? `01_kv_cache_worked.ipynb` is the answer key.

In [ ]:
# numpy + matplotlib (both preinstalled on Colab); no GPU, no torch. The next lines make `import kerncore`
# work from a checkout without installing anything: kernel-core sits next to this folder.
import os, sys, pathlib, math, time
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")   # tiny matrices: one BLAS thread is faster, and steady
os.environ.setdefault("OMP_NUM_THREADS", "1")        # on a shared CPU (set before numpy is imported)
_core = pathlib.Path.cwd().resolve().parent / "kernel-core"
if (_core / "kerncore").is_dir() and str(_core) not in sys.path:
    sys.path.insert(0, str(_core))
import numpy as np
import matplotlib.pyplot as plt
from kerncore import kv            # random weights, a reference copy of the model, the size formulas

rng = np.random.default_rng(0)
print("numpy + kerncore ready")

## Attention — Blanks A and B

The cache lives here. **A**: join this step's K/V onto the remembered ones.
**B**: stop each query from seeing tokens ahead of it.

In [ ]:
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


class CausalSelfAttention:
    def __init__(self, p, n_heads):
        self.wq, self.wk, self.wv, self.wo = p["wq"], p["wk"], p["wv"], p["wo"]
        self.n_heads = n_heads
        self.head_dim = self.wq.shape[1] // n_heads

    def forward(self, x, cache=None):
        T, C = x.shape
        split = lambda t: t.reshape(T, self.n_heads, self.head_dim).transpose(1, 0, 2)
        q, k, v = split(x @ self.wq), split(x @ self.wk), split(x @ self.wv)   # (n_heads, T, head_dim)

        past_len = 0
        if cache is not None:
            past_k, past_v = cache
            past_len = past_k.shape[1]
            # ---- BLANK A: grow the cache -------------------------------------
            # Glue this step's k and v onto the remembered past_k / past_v.
            # They are shaped (n_heads, seq, head_dim); join along the seq axis,
            # oldest first.
            # YOUR CODE HERE: k = np.concatenate([...], axis=?) and the same for v
            raise NotImplementedError("BLANK A: grow the cache (replace this line with the two np.concatenate lines)")
            # ------------------------------------------------------------------
        new_cache = (k, v)

        att = q @ k.transpose(0, 2, 1) / math.sqrt(self.head_dim)   # (n_heads, T, past+T)

        # Absolute positions: query i is at past_len+i; key j is at j.
        q_pos = np.arange(past_len, past_len + T)[:, None]
        k_pos = np.arange(past_len + T)[None, :]
        # ---- BLANK B: causal mask --------------------------------------------
        # A query must NOT see keys that lie in its future. Set those scores to
        # -inf so softmax gives them zero weight. (Which comparison of k_pos and
        # q_pos marks a key as "in the future"?)
        # YOUR CODE HERE: att = np.where(<condition>, -np.inf, att)
        raise NotImplementedError("BLANK B: the causal mask (replace this line with the np.where)")
        # ----------------------------------------------------------------------
        att = softmax(att)

        y = (att @ v).transpose(1, 0, 2).reshape(T, C)
        return y @ self.wo, new_cache

In [ ]:
class MiniLM:
    """Token + position embedding -> attention layers (residual) -> output head. One cache per layer (a list)."""
    def __init__(self, params):
        self.p = params
        self.layers = [CausalSelfAttention(lp, params["n_heads"]) for lp in params["layers"]]
    def forward(self, idx, caches=None):
        past_len = 0 if caches is None else caches[0][0].shape[1]
        x = self.p["emb"][idx] + self.p["pos"][past_len:past_len + len(idx)]
        new_caches = []
        for i, layer in enumerate(self.layers):
            y, c = layer.forward(x, None if caches is None else caches[i])
            x = x + y
            new_caches.append(c)
        return x @ self.p["head"], new_caches

## Generation — Blank C

Naive is given. In cached generation, decode must feed **one** token plus the cache.

In [ ]:
def generate_naive(model, prompt, n_new):        # given, for reference
    idx, last_logits = list(prompt), []
    for _ in range(n_new):
        logits, _ = model.forward(np.array(idx))
        last_logits.append(logits[-1])
        idx.append(int(logits[-1].argmax()))
    return np.array(idx), np.array(last_logits)

In [ ]:
def generate_cached(model, prompt, n_new):
    logits, caches = model.forward(np.array(prompt))         # prefill: whole prompt, no cache
    last_logits = [logits[-1]]
    idx = list(prompt) + [int(logits[-1].argmax())]
    for _ in range(n_new - 1):
        # ---- BLANK C: one decode step ------------------------------------
        # Feed ONLY the newest token plus the cache, and capture the updated
        # cache. (What are the two arguments to model.forward here?)
        # YOUR CODE HERE: logits, caches = model.forward(np.array([idx[-1]]), ...)   pass the cache too
        raise NotImplementedError("BLANK C: one decode step (replace this line with the model call)")
        # ------------------------------------------------------------------
        last_logits.append(logits[-1])
        idx.append(int(logits[-1].argmax()))
    return np.array(idx), np.array(last_logits)

## Tests for A + C

Test 1 checks the full loop agrees with naive, token for token and logit for logit. Test 2 isolates a single
decode step against a full forward pass, then feeds a two-token chunk on top of the cache (what chunked prefill
does), which also checks that the cache is kept oldest-first.

In [ ]:
# --- Test 1: caching preserves output (checks BLANK A + BLANK C) --------------
model  = MiniLM(kv.random_params(256, 128, 4, 4, seed=0))
prompt = rng.integers(0, 256, 16)
a, la = generate_naive(model,  prompt, 32)
b, lb = generate_cached(model, prompt, 32)
assert np.all(np.isfinite(lb)), "NaN or inf in the logits -> does the mask (B) hide a query's own key?"
assert np.array_equal(a, b), "cached tokens != naive tokens -> check the cache concat (A), the mask (B), the decode call (C)"
assert np.abs(la - lb).max() <= 1e-9, "cached logits drift from naive -> check the cache concat (A) and the decode call (C)"
print("Test 1 PASSED - cached matches naive:", b[-6:].tolist())

In [ ]:
# --- Test 2: one decode step == last position of a full forward (checks BLANK A)
m   = MiniLM(kv.random_params(256, 64, 4, 2, seed=1))
seq = rng.integers(0, 256, 3)
full, _    = m.forward(seq)                  # feed all 3 at once
_, caches  = m.forward(seq[:2])              # prefill first 2
step, _    = m.forward(seq[2:], caches)      # decode the 3rd using the cache
assert np.allclose(full[-1], step[0], atol=1e-10), "decode-with-cache disagrees with full forward"

# A two-token chunk on top of a cache (chunked prefill) also needs the cache in time order:
seq = rng.integers(0, 256, 6)
full, full_caches = m.forward(seq)
_, caches = m.forward(seq[:4])
chunk, grown = m.forward(seq[4:], caches)
assert np.allclose(full[4:], chunk, atol=1e-10), "a chunk on top of the cache disagrees -> is the cache oldest-first?"
assert all(np.allclose(k1, k2) for (k1, _), (k2, _) in zip(grown, full_caches)),     "the grown cache differs from a full forward's K -> is it oldest-first (A), and does the mask (B) hide only the future?"
print("Test 2 PASSED - decode step and a 2-token chunk reproduce the full-sequence result")

## Test for B

Causality has a clean signature: editing a *future* token must leave *earlier* outputs
untouched. If your mask is missing or points the wrong way, this fails.

In [ ]:
# --- Test 3: attention is causal (checks BLANK B) ----------------------------
# If the mask is right, changing a FUTURE token cannot alter an EARLIER output.
m  = MiniLM(kv.random_params(256, 64, 4, 2, seed=2))
x1 = rng.integers(0, 256, 6)
x2 = x1.copy(); x2[-1] = (x2[-1] + 1) % 256        # change only the last token
l1, _ = m.forward(x1); l2, _ = m.forward(x2)
assert np.all(np.isfinite(l1)), "NaN or inf in the logits -> does the mask hide a query's own key?"
assert np.allclose(l1[:-1], l2[:-1], atol=1e-10), \
    "changing a future token changed the past -> causal mask is missing/wrong"
print("Test 3 PASSED - future tokens do not leak into the past")

## Memory — Blank D

The formula from the primer. Test 4 checks it against the Llama 3 8B number (128 KiB per token) and against
`kerncore.kv.kv_cache_bytes` for a few other shapes and batch sizes.

In [ ]:
def kv_cache_bytes(n_layers, n_kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    # ---- BLANK D: cache size -------------------------------------------------
    # Total bytes = (K and V, so a factor of 2) x layers x kv_heads x head_dim
    #               x seq_len x batch x bytes_per_value.
    # YOUR CODE HERE: return ...
    raise NotImplementedError("BLANK D: the cache-size formula (replace this line with the return)")
    # --------------------------------------------------------------------------

# --- Test 4: matches the Llama 3 8B figure from the primer, and kerncore's formula --------
per_tok = kv_cache_bytes(32, 8, 128, seq_len=1)
assert per_tok == 131072, f"expected 131072 bytes/token, got {per_tok}"
for args in ((32, 8, 128, 8192, 32, 2), (40, 40, 128, 2048, 1, 2), (32, 8, 128, 1000, 3, 1)):
    assert kv_cache_bytes(*args) == kv.kv_cache_bytes(*args), f"wrong for {args}: check every factor, batch included"
print(f"Test 4 PASSED - {kv.fmt_bytes(per_tok)} per token")
_b = kv_cache_bytes(32, 8, 128, 128_000)
print(f"1 user @ 128,000 tokens : {kv.fmt_bytes(_b)}")
_b = kv_cache_bytes(32, 8, 128, 8192, batch=32)
print(f"32 users @ 8,192 tokens : {kv.fmt_bytes(_b)}")

## Optional: the cost curve

Once all four tests pass, run this to see the decode-phase story — naive rising with
the prefix, cached rising only with the cache.

In [ ]:
# Optional: once the tests pass, watch the decode-phase cost curve. kerncore's copy of the model counts
# multiply-adds, so the curves are exact; the wall-clock line is this machine's.
counted = kv.TinyDecoder(kv.random_params(256, 128, 4, 4, seed=1))
p = rng.integers(0, 256, 32)
_, _, naive_c  = kv.generate_naive(counted, p, 128)
_, _, cached_c = kv.generate_cached(counted, p, 128)
plt.figure(figsize=(7, 4))
plt.plot(kv.per_step_costs(naive_c) / 1e6, label="naive")
plt.plot(range(1, 128), kv.per_step_costs(cached_c)[1:] / 1e6, label="cached")
plt.xlabel("generation step"); plt.ylabel("million multiply-adds / token"); plt.legend()
plt.title("naive grows with the prefix; cached grows only with the cache"); plt.tight_layout(); plt.show()

big = MiniLM(kv.random_params(256, 128, 4, 4, seed=1))
t0 = time.perf_counter(); generate_naive(big, p, 128); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); generate_cached(big, p, 128); t_cached = time.perf_counter() - t0
print(f"your generate_cached vs generate_naive on this CPU: {t_cached:.2f} s vs {t_naive:.2f} s")

## Done

If all four tests pass, you have a correct KV cache: **A** and **C** make it *fast*
(linear, not quadratic), **B** keeps it *correct* (causal), and **D** is why it is the
memory bottleneck that dominates real GPU serving.